<a href="https://colab.research.google.com/github/Avhi-kr/DL-GenAI-Practice/blob/main/NN_module_using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [8]:
torch.manual_seed(42)

In [9]:
device=torch.device('cuda' if torch.cuda.is_available else 'cpu')

In [10]:
device

device(type='cuda')

In [11]:
df=pd.read_csv('/content/fashion-mnist_test.csv.zip')

In [12]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,9,8,...,103,87,56,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,14,53,99,...,0,0,0,0,63,53,31,0,0,0
3,2,0,0,0,0,0,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
x=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [14]:
y

array([0, 1, 2, ..., 8, 8, 1])

In [15]:
x=x/255

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [17]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [18]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset=CustomDataset(x_test,y_test)

In [19]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)

In [20]:
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=True)

In [21]:
class my_NN(nn.Module):
  def __init__(self,no_features):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(no_features,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,10)
    )
  def forward(self,features):
    return self.model(features)

In [22]:
learning_rate=0.1
epochs=100
criterion=nn.CrossEntropyLoss()

In [23]:
model=my_NN(x_train.shape[1])
model=model.to(device)

In [24]:
optimizer=optim.SGD(model.parameters(),lr=learning_rate)

In [25]:
train_loader

In [26]:
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_features,batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    fwd=model(batch_features)
    batch_loss=criterion(fwd,batch_labels) # Using 'criterion' for the loss function
    batch_loss.backward()
    optimizer.zero_grad()
    optimizer.step()
    total_epoch_loss += batch_loss.item()

In [27]:
model.eval()
with torch.no_grad():
  total=0
  correct=0
  for batch_features,batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output=model(batch_features)
    _, predicted=torch.max(output,1)
    total=total+batch_labels.shape[0]
    correct+=(predicted==batch_labels).sum().item()
  print(correct/total)

0.086


In [28]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.086
